# Nemotron Voice Agent

Nemotron Voice Agent Blueprint provides a comprehensive, end-to-end voice agent built with NVIDIA Nemotron state-of-the-art open models, as NVIDIA NIM for acceleration and scaling. It is designed to guide developers through the creation of a cascaded pipeline, integrating Nemotron ASR, LLM, and TTS, while solving for the complexities of streaming, interruptible conversations. Clone it, swap in your own logic, and deploy a working voice AI prototype in hours.

Built on the open-source [Pipecat framework](https://github.com/pipecat-ai/pipecat) and leveraging [NVIDIA NIM microservices](https://build.nvidia.com/), this example helps teams accelerate the deployment of high-performance voice AI solutions.

**GitHub Repository:** [NVIDIA-AI-Blueprints/nemotron-voice-agent](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent)

## Why This Blueprint

- **Sub-second E2E latency:** sub-second end-to-end latency with support for multiple concurrent streams, designed for production scale.
- **Fully open models:** Nemotron Streaming and Parakeet ASR, Magpie TTS and Nemotron LLM models. Swap any component, self-host, no lock-in.
- **Interruption Handling:** Voice Activity Detection (VAD) and End-of-Utterance (EOU) logic to guide the agent on exactly when to start and stop speaking, ensuring a natural conversational flow.
- **Multilingual Capabilities:** native support for multiple languages provided by NVIDIA Magpie TTS and Multilingual ASR.
- **Multimodal Understanding:** reason over speech and vision together, analyzing live camera input and uploaded media (images, documents) within a single conversation, powered by Nemotron Omni.
- **Multi-Agent and Tool Calling:** orchestrate cooperating agents that invoke external tools and functions for task-oriented workflows, while decoupling reasoning from response generation for lower perceived latency.
- **Edge Support:** deploy anywhere, from cloud and workstation to DGX Spark and edge devices like Jetson Thor, using self-contained deployment recipes.

---

## Architecture

![Nemotron Voice Agent Architecture](https://raw.githubusercontent.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/main/docs/images/arch.png)

---

## Available Examples

Review the examples below and select the one that best matches your use case. Set `EXAMPLE_PROFILE` in the **Setup** section to the corresponding profile value. The default is `generic-assistant/workstation`.

| Example | Profile value | Description |
|---------|--------------|-------------|
| [Generic Assistant](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/generic/README.md) | `generic-assistant/workstation` | Baseline English cascaded pipeline: Nemotron ASR + LLM + Magpie TTS. Best starting point. |
| [Multilingual Assistant](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/multilingual/README.md) | `multilingual-assistant/workstation` | Multilingual ASR and TTS with a fixed language per session. |
| [Nemotron Omni Assistant](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/omni_assistant/README.md) | `omni-assistant/workstation` | Single Nemotron Omni model replaces ASR + LLM stages. Multimodal (audio/vision). |
| [Omni Subagents](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/omni_assistant_subagents/README.md) | `omni-assistant-subagents/workstation` | Multi-agent Omni pipeline for richer audio/video/webcam understanding. |
| [Frontend/Backend Agent](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/frontend_backend_agent/README.md) | `frontend-backend-agent/workstation` | Fast frontend LLM + specialized backend agent. Add voice to an existing text/agentic backend. |

---

## 1. Prerequisites

Before running this notebook, confirm your environment has:

| Requirement | Details |
|-------------|--------|
| **GPU** | Single NVIDIA GPU with ≥ 72 GB VRAM (H100, RTX 6000 Pro, etc.) |
| **Docker** | Docker Engine with NVIDIA GPU support and Docker Compose v2.20+ |
| **NVIDIA API Key** | Required for NGC container registry and NVIDIA-hosted NIM APIs. Get one at [build.nvidia.com/settings/api-keys](https://build.nvidia.com/settings/api-keys) |
| **Network ports** | Port `7860` for the web app; `3478` and `49160–49200` UDP/TCP for WebRTC TURN |

---

## 2. Setup

### 2a. Clone the Repository

This cell clones the Nemotron Voice Agent repository if it is not already present.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent.git"
CLONE_DIR = Path(os.path.expanduser("~/nemotron-voice-agent"))

if not CLONE_DIR.exists():
    print(f"Cloning {REPO_URL} → {CLONE_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    print("Clone complete.")
else:
    print(f"Repository already present at {CLONE_DIR}. Pulling latest changes.")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

REPO_ROOT = CLONE_DIR
print(f"Repo root: {REPO_ROOT}")

### 2b. Configure API Keys and Deployment Settings

Enter your credentials below. All fields in one place — run the cell and you will be prompted for any missing values.

In [ ]:
import getpass

# ── Example to deploy ──────────────────────────────────────────────────────────
# Options:
#   generic-assistant/workstation          ← default, English cascaded pipeline
#   multilingual-assistant/workstation
#   omni-assistant/workstation
#   omni-assistant-subagents/workstation
#   frontend-backend-agent/workstation
EXAMPLE_PROFILE = os.environ.get("EXAMPLE_PROFILE", "generic-assistant/workstation")

# ── Credentials (prompted if not set) ─────────────────────────────────────────
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
# Required for Omni-based examples (omni-assistant, omni-assistant-subagents)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Advanced settings ──────────────────────────────────────────────────────────
PIPELINE_APP_PORT = os.environ.get("PIPELINE_APP_PORT", "7860")
PIPELINE_TLS = "false"  # Brev Secure Links proxy HTTP → set to "true" for direct HTTPS
RESET_VOLUMES = False  # Set True to wipe Docker/NIM model caches and start fresh
AUTO_DETECT_TURN = True  # Auto-detect public IP for TURN URL

_omni_profile = EXAMPLE_PROFILE.startswith("omni-")

print(f"Example profile : {EXAMPLE_PROFILE}")
print(f"App port        : {PIPELINE_APP_PORT}")
print(f"TLS             : {PIPELINE_TLS}")
print()

# Prompt for any missing credentials
if not NVIDIA_API_KEY or NVIDIA_API_KEY in ("nvapi-...", ""):
    NVIDIA_API_KEY = getpass.getpass("🔑 NVIDIA_API_KEY (required): ").strip()
if not NVIDIA_API_KEY:
    raise RuntimeError("NVIDIA_API_KEY is required to pull NGC containers and call NIM APIs.")

if not HF_TOKEN:
    _prompt = (
        "🤗 HF_TOKEN (required for this Omni-based example): "
        if _omni_profile
        else "🤗 HF_TOKEN (required for Omni-based examples — press Enter to skip): "
    )
    _hf = getpass.getpass(_prompt).strip()
    if _hf:
        HF_TOKEN = _hf

if _omni_profile and not HF_TOKEN:
    raise RuntimeError(
        f"HF_TOKEN is required for the '{EXAMPLE_PROFILE}' profile. "
        "Get a token at https://huggingface.co/settings/tokens"
    )

print("Credentials configured.")

### 2c. Initialize Utilities and `.env`

In [ ]:
import json
import secrets
import subprocess
import time
import urllib.request

ENV_PATH = REPO_ROOT / ".env"
ENV_EXAMPLE_PATH = REPO_ROOT / ".env.example"
PROFILES = [EXAMPLE_PROFILE, "turn"]
DOCKER = ["docker"]
COMPOSE = [*DOCKER, "compose"]
COMPOSE_PROGRESS = [*DOCKER, "compose", "--progress", "plain"]
for _p in PROFILES:
    COMPOSE.extend(["--profile", _p])
    COMPOSE_PROGRESS.extend(["--profile", _p])


def redact(text: str) -> str:
    """Replace secret values with a redaction marker before printing."""
    for v in (NVIDIA_API_KEY, HF_TOKEN):
        if v:
            text = text.replace(v, "***REDACTED***")
    return text


def run(cmd, *, check=True, input_text=None, stream=False):
    """Run a shell command from the repo root and print redacted output."""
    print("$ " + redact(" ".join(cmd)))
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            stdin=subprocess.PIPE if input_text else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if input_text and proc.stdin:
            proc.stdin.write(input_text)
            proc.stdin.close()
        out = []
        for line in proc.stdout:
            out.append(line)
            stripped = line.rstrip()
            if not stripped.endswith("Pulling fs layer 0B"):
                print(redact(stripped))
        proc.wait()
        stdout = "".join(out)
        if check and proc.returncode != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd, stdout, "")
        return subprocess.CompletedProcess(cmd, proc.returncode, stdout, "")
    proc = subprocess.run(cmd, cwd=REPO_ROOT, input=input_text, text=True, capture_output=True)
    if proc.stdout:
        print(redact(proc.stdout.rstrip()))
    if proc.stderr:
        print(redact(proc.stderr.rstrip()))
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, proc.stdout, proc.stderr)
    return proc


def fetch_text(url, timeout=10):
    """Fetch a URL and return the response body as text."""
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode("utf-8", errors="replace")


def detect_public_ip():
    """Return the public IP of this instance."""
    return urllib.request.urlopen("https://ifconfig.me", timeout=10).read().decode().strip()


def read_env(path):
    """Read an env file into a list of lines."""
    return path.read_text().splitlines() if path.exists() else []


def get_env(lines, key):
    """Return the value for a key from parsed env lines."""
    for line in lines:
        s = line.strip()
        if s.startswith(f"{key}="):
            return s.split("=", 1)[1]
    return ""


def set_env(lines, key, value):
    """Set or append a key=value in parsed env lines."""
    replacement = f"{key}={value}"
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith(f"{key}=") or s.startswith(f"# {key}=") or s.startswith(f"#{key}="):
            lines[i] = replacement
            return lines
    lines.append(replacement)
    return lines


# Bootstrap .env from example if missing
if not ENV_PATH.exists():
    ENV_PATH.write_text(ENV_EXAMPLE_PATH.read_text())

lines = read_env(ENV_PATH)

TURN_USERNAME = os.environ.get("TURN_USERNAME") or get_env(lines, "TURN_USERNAME") or "admin"
TURN_PASSWORD = os.environ.get("TURN_PASSWORD") or get_env(lines, "TURN_PASSWORD") or ""
TURN_URL = os.environ.get("TURN_URL") or get_env(lines, "TURN_URL") or ""

if not TURN_PASSWORD or TURN_PASSWORD == "admin":
    TURN_PASSWORD = secrets.token_urlsafe(32)
    print("Generated random TURN_PASSWORD.")

if AUTO_DETECT_TURN and (not TURN_URL or "brevlab.com" in TURN_URL):
    _ip = detect_public_ip()
    TURN_URL = f"turn:{_ip}:3478"
    print(f"Auto-detected TURN_URL: {TURN_URL}")

env_values = {
    "NVIDIA_API_KEY": NVIDIA_API_KEY,
    "HF_TOKEN": HF_TOKEN,
    "TURN_USERNAME": TURN_USERNAME,
    "TURN_PASSWORD": TURN_PASSWORD,
    "TURN_URL": TURN_URL,
    "PIPELINE_APP_PORT": PIPELINE_APP_PORT,
    "PIPELINE_TLS": PIPELINE_TLS,
}
for k, v in env_values.items():
    if v:
        lines = set_env(lines, k, v)
ENV_PATH.write_text("\n".join(lines).rstrip() + "\n")
os.environ.update({k: v for k, v in env_values.items() if v})
print(f"\n.env written: {ENV_PATH}")
print(f"Profiles     : {', '.join(PROFILES)}")

---

## 3. Preflight Checks

Verify Docker, GPU visibility, and disk space before pulling containers.

In [ ]:
run([*DOCKER, "--version"])
run([*DOCKER, "compose", "version"])
run([*DOCKER, "ps"])
run([*DOCKER, "info", "--format", "DockerRootDir={{.DockerRootDir}} runtime={{.DefaultRuntime}}"], check=False)
run(["df", "-h", str(REPO_ROOT)], check=False)
run(["nvidia-smi"], check=False)

---

## 4. Deploy

Logs into the NGC container registry, pulls and builds images, then starts all services. **First run takes 30–60 minutes** while NIM model weights download. Subsequent runs start in seconds (weights are cached in Docker volumes).

Safe to re-run — containers are recreated, model-cache volumes are preserved unless `RESET_VOLUMES=True`.

In [ ]:
# NGC login
run([*DOCKER, "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"], input_text=NVIDIA_API_KEY + "\n")

# Tear down any existing deployment
down_cmd = [*COMPOSE, "down", "--remove-orphans"]
if RESET_VOLUMES:
    down_cmd.append("-v")
    print("RESET_VOLUMES=True — model cache volumes will be deleted.")
run(down_cmd, check=False)

# Pull and start (--quiet-pull suppresses per-layer download noise)
run([*COMPOSE_PROGRESS, "up", "-d", "--build", "--remove-orphans", "--quiet-pull"], stream=True)
run([*COMPOSE, "ps"])

---

## 5. Accessing the Application

### Health check

The cell below polls until the app is ready (up to 5 minutes). NIM sidecars (ASR, LLM, TTS) may take longer on first boot while loading model weights — this is normal.

In [ ]:
scheme = "http" if PIPELINE_TLS.lower() == "false" else "https"
health_url = f"{scheme}://localhost:{PIPELINE_APP_PORT}/health"
ice_url = f"{scheme}://localhost:{PIPELINE_APP_PORT}/api/ice-servers"

for attempt in range(1, 31):
    try:
        fetch_text(health_url, timeout=5)
        print(f"✅ App is healthy: {health_url}")
        break
    except Exception as exc:
        if attempt == 30:
            raise
        print(f"Waiting for app ({attempt}/30): {exc}")
        time.sleep(10)

try:
    ice = json.loads(fetch_text(ice_url, timeout=10))
    print("ICE / TURN config:")
    print(json.dumps(ice, indent=2))
except Exception as exc:
    print(f"ICE check failed (non-fatal): {exc}")

### Open the Web UI

**Brev Secure Link** (remote access via Brev proxy):
1. In the Brev console, go to **Using Secure Links → Share a Service**.
2. If not already created, create a Secure Link for port `7860`.
3. Open the generated `*.brevlab.com` URL in your browser.

Once connected, you will see the Nemotron Voice Agent interface:

![Nemotron Voice Agent UI](../docs/images/ui.png)

Press **Connect** to start a voice conversation.

### Tips

- **Audio:** use a wired headset for best results.
- **Warm-up:** the first voice turn may be slower while NIM GPU sidecars finish loading. Subsequent turns are much faster.
- **Logs:** run `docker compose logs -f` in the repo root to tail all service logs.

---

### Troubleshooting

**WebRTC not connecting?**

Expose the TURN ports in the Brev console **Using Ports** section:
- Port `3478` (TCP + UDP): TURN control
- Range `49160-49200` (UDP): TURN relay

> The notebook auto-detects the instance public IP for `TURN_URL`. Do **not** use the Brev Secure Link hostname as `TURN_URL`. TURN requires a direct UDP/TCP path to the instance IP.